# Age Classification — b23cm1033

**Phase 2 — Improved b23cm1054 approach**

Improvements over rank 4 (b23cm1054):
- EMA weight averaging (+1-2%)
- Stronger augmentation (GaussianBlur, RandomErasing, stronger ColorJitter)
- Test Time Augmentation at inference (+1-2%)
- More Stage 2 epochs (60 vs 50)
- Label smoothing kept (0.1)


In [21]:
import os, csv, math, sys, random, copy
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')


Device: cuda


In [22]:
import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', 'open_clip_torch', '--quiet'], check=True)
print('open_clip ready.')


open_clip ready.


In [ ]:
# Config
ROLL_NO       = 'b23cm1033'
DATA_ROOT     = 'dataset'
TRAIN_DIR     = os.path.join(DATA_ROOT, 'train')
VALID_DIR     = os.path.join(DATA_ROOT, 'valid')
VALID_CSV     = os.path.join(DATA_ROOT, 'valid_labels.csv')

IMG_SIZE      = 224

BATCH_SIZE    = 64
NUM_WORKERS   = 0
CLIP_FEAT_DIM = 512       # ViT-B/32
NUM_EP_S1     = 150       # Stage 1: distillation
NUM_EP_S2     = 80        # Stage 2: classification (improved: 60 vs rank4's 50)
EMA_DECAY     = 0.9997    # NEW: EMA weight averaging
BACKBONE_FILE = 'backbone_pretrain.pth'
CLIP_CACHE    = 'clip_feats_vitb32.pt'

print(f'Stage 1: {NUM_EP_S1} | Stage 2: {NUM_EP_S2} | EMA: {EMA_DECAY}')


Stage 1: 150 | Stage 2: 80 | EMA: 0.9997


In [24]:
# Model — _Net at module level so torch.save() works
class _Net(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        base = models.resnet18(weights=None)
        self.backbone   = nn.Sequential(*list(base.children())[:-1])
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(256, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(64, 2),
        )
    def extract_features(self, x):
        return self.backbone(x).flatten(1)
    def forward(self, x):
        return self.classifier(self.extract_features(x))

def build_model(num_classes=2, dropout=0.3):
    return _Net(dropout=dropout)

_t = build_model()
assert _t(torch.randn(2,3,224,224)).shape == (2,2)
print(f'Model OK | params: {sum(p.numel() for p in _t.parameters()):,}')
del _t

Model OK | params: 11,325,058


In [25]:
# EMA — slow moving weight copy, generalizes better to hidden test set
class EMA:
    def __init__(self, model, decay=0.9997):
        self.decay   = decay
        self.shadow  = {k: v.clone().float() for k, v in model.state_dict().items()}
        self._backup = {}

    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            self.shadow[k] = self.decay * self.shadow[k] + (1 - self.decay) * v.float()

    def apply_to(self, model):
        model.load_state_dict({k: v.to(next(model.parameters()).device)
                               for k, v in self.shadow.items()})

    def store(self, model):
        self._backup = {k: v.clone() for k, v in model.state_dict().items()}

    def restore(self, model):
        model.load_state_dict(self._backup)

print('EMA ready.')


EMA ready.


In [26]:
import open_clip

def get_clip_embeddings(train_root, val_dir, val_csv):
    print('Loading CLIP ViT-B/32...')
    clip_mdl, _, preproc = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
    clip_mdl = clip_mdl.to(DEVICE).eval()
    for p in clip_mdl.parameters(): p.requires_grad = False

    feats = []
    for cls in [0, 1]:
        folder = os.path.join(train_root, str(cls))
        files  = sorted(f for f in os.listdir(folder) if f.endswith('.png'))
        print(f'  train class {cls}: {len(files)} images')
        for i in range(0, len(files), 32):
            chunk = files[i:i+32]
            imgs  = [preproc(Image.open(os.path.join(folder,f)).convert('RGB')) for f in chunk]
            with torch.no_grad():
                emb = F.normalize(clip_mdl.encode_image(torch.stack(imgs).to(DEVICE)).float(), dim=-1)
            feats.append(emb.cpu())

    with open(val_csv) as f:
        val_files = [r['image'] for r in csv.DictReader(f)]
    print(f'  val: {len(val_files)} images')
    for i in range(0, len(val_files), 32):
        chunk = val_files[i:i+32]
        imgs  = [preproc(Image.open(os.path.join(val_dir,fn)).convert('RGB')) for fn in chunk]
        with torch.no_grad():
            emb = F.normalize(clip_mdl.encode_image(torch.stack(imgs).to(DEVICE)).float(), dim=-1)
        feats.append(emb.cpu())

    all_feats = torch.cat(feats, dim=0)
    print(f'Total embeddings: {all_feats.shape}')
    del clip_mdl; torch.cuda.empty_cache()
    return all_feats

if os.path.exists(CLIP_CACHE):
    clip_feats = torch.load(CLIP_CACHE, map_location='cpu', weights_only=False)
    print(f'Loaded cached CLIP features: {clip_feats.shape}')
else:
    clip_feats = get_clip_embeddings(TRAIN_DIR, VALID_DIR, VALID_CSV)
    torch.save(clip_feats, CLIP_CACHE)
    print(f'Saved: {CLIP_CACHE}')


Loaded cached CLIP features: torch.Size([18466, 512])


In [27]:
class FaceFullSet(Dataset):
    def __init__(self, train_root, val_dir, val_csv, tfm=None):
        self.tfm  = tfm; self.data = []
        for lbl in [0, 1]:
            d = os.path.join(train_root, str(lbl))
            for fn in sorted(os.listdir(d)):
                if fn.endswith('.png'): self.data.append((os.path.join(d, fn), lbl))
        with open(val_csv) as f:
            for r in csv.DictReader(f):
                self.data.append((os.path.join(val_dir, r['image']), int(r['label'])))
        print(f'Dataset: {len(self.data)} images')
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        path, lbl = self.data[idx]
        img = Image.open(path).convert('RGB')
        if self.tfm: img = self.tfm(img)
        return img, lbl, idx

class ValidDataset(Dataset):
    def __init__(self, img_dir, csv_path, transform=None):
        self.img_dir = img_dir; self.transform = transform; self.samples = []
        with open(csv_path) as f:
            for row in csv.DictReader(f):
                self.samples.append((row['image'], int(row['label'])))
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        fname, label = self.samples[idx]
        img = Image.open(os.path.join(self.img_dir, fname)).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, label

# Stage 1 augmentation — simple, for distillation
aug_s1 = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

# Stage 2 augmentation — IMPROVED vs rank 4
aug_s2 = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.1),
    transforms.RandomGrayscale(p=0.15),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.1,2.0))], p=0.3),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.2)),
])

val_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

# TTA transforms — 3 views averaged at inference
tta_tfms = [
    transforms.Compose([transforms.Resize((IMG_SIZE,IMG_SIZE)), transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])]),
    transforms.Compose([transforms.Resize((IMG_SIZE,IMG_SIZE)), transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(), transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])]),
    transforms.Compose([transforms.Resize((IMG_SIZE+16,IMG_SIZE+16)), transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(), transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])]),
]
print('Transforms + TTA ready.')


Transforms + TTA ready.


In [28]:
def lr_schedule(ep, warmup, peak_lr, total):
    if ep < warmup:
        return peak_lr * (ep + 1) / warmup
    t = (ep - warmup) / (total - warmup)
    return 1e-6 + 0.5 * (peak_lr - 1e-6) * (1 + math.cos(math.pi * t))

@torch.no_grad()
def evaluate(mdl, loader):
    mdl.eval()
    correct, total = 0, 0
    for imgs, lbls in loader:
        imgs = imgs.to(DEVICE)
        lbls = lbls.clone().detach().long().to(DEVICE)
        correct += (mdl(imgs).argmax(1) == lbls).sum().item()
        total   += lbls.size(0)
    return correct / total * 100

@torch.no_grad()
def evaluate_tta(mdl, img_dir, csv_path):
    mdl.eval()
    samples = []
    with open(csv_path) as f:
        for row in csv.DictReader(f):
            samples.append((row['image'], int(row['label'])))
    correct, total = 0, 0
    for fname, label in samples:
        img = Image.open(os.path.join(img_dir, fname)).convert('RGB')
        logits_sum = None
        for t in tta_tfms:
            inp = t(img).unsqueeze(0).to(DEVICE)
            logits = mdl(inp)
            logits_sum = logits if logits_sum is None else logits_sum + logits
        correct += int(logits_sum.argmax(1).item() == label)
        total   += 1
    return correct / total * 100


In [29]:
# ── STAGE 1: CLIP Feature Distillation ───────────────────────────────────────
if os.path.exists(BACKBONE_FILE):
    print(f'Found {BACKBONE_FILE} — skipping Stage 1.')
    net = torch.load(BACKBONE_FILE, map_location=DEVICE, weights_only=False)
    net.to(DEVICE)
else:
    print(f'Stage 1: {NUM_EP_S1} epochs of CLIP distillation...')
    net  = build_model().to(DEVICE)
    proj = nn.Sequential(
        nn.Linear(512, 512),
        nn.GELU(),
        nn.Linear(512, CLIP_FEAT_DIM),
    ).to(DEVICE)

    ds1 = FaceFullSet(TRAIN_DIR, VALID_DIR, VALID_CSV, tfm=aug_s1)
    dl1 = DataLoader(ds1, batch_size=BATCH_SIZE, shuffle=True,
                     drop_last=True, num_workers=NUM_WORKERS)
    opt1 = optim.Adam(list(net.parameters()) + list(proj.parameters()),
                      lr=1e-3, weight_decay=1e-4)

    for ep in range(NUM_EP_S1):
        net.train(); proj.train()
        cur_lr = lr_schedule(ep, 3, 1e-3, NUM_EP_S1)
        for g in opt1.param_groups: g['lr'] = cur_lr
        run_loss, n = 0.0, 0
        for imgs, _, idxs in dl1:
            imgs    = imgs.to(DEVICE)
            targets = clip_feats[idxs].to(DEVICE)
            opt1.zero_grad()
            out  = F.normalize(proj(net.extract_features(imgs)), dim=-1)
            loss = (1 - (out * targets).sum(dim=-1)).mean()
            loss.backward()
            nn.utils.clip_grad_norm_(list(net.parameters())+list(proj.parameters()), 1.0)
            opt1.step()
            run_loss += loss.item() * imgs.size(0); n += imgs.size(0)
        if (ep+1) % 10 == 0 or ep == 0:
            print(f'  S1 Epoch {ep+1:3d}/{NUM_EP_S1}  Loss: {run_loss/n:.4f}  LR: {cur_lr:.2e}')

    del proj
    torch.save(net, BACKBONE_FILE)
    print(f'Stage 1 done. Saved: {BACKBONE_FILE}')


Stage 1: 150 epochs of CLIP distillation...
Dataset: 18466 images
  S1 Epoch   1/150  Loss: 0.1959  LR: 3.33e-04
  S1 Epoch  10/150  Loss: 0.1251  LR: 9.96e-04
  S1 Epoch  20/150  Loss: 0.1108  LR: 9.71e-04
  S1 Epoch  30/150  Loss: 0.1035  LR: 9.25e-04
  S1 Epoch  40/150  Loss: 0.0999  LR: 8.59e-04
  S1 Epoch  50/150  Loss: 0.0966  LR: 7.77e-04
  S1 Epoch  60/150  Loss: 0.0941  LR: 6.83e-04
  S1 Epoch  70/150  Loss: 0.0912  LR: 5.80e-04
  S1 Epoch  80/150  Loss: 0.0884  LR: 4.74e-04
  S1 Epoch  90/150  Loss: 0.0854  LR: 3.69e-04
  S1 Epoch 100/150  Loss: 0.0818  LR: 2.69e-04
  S1 Epoch 110/150  Loss: 0.0787  LR: 1.81e-04
  S1 Epoch 120/150  Loss: 0.0755  LR: 1.07e-04
  S1 Epoch 130/150  Loss: 0.0728  LR: 5.05e-05
  S1 Epoch 140/150  Loss: 0.0708  LR: 1.47e-05
  S1 Epoch 150/150  Loss: 0.0700  LR: 1.11e-06
Stage 1 done. Saved: backbone_pretrain.pth


In [30]:
# ── STAGE 2: Classification + EMA (IMPROVED) ─────────────────────────────────
full_ds    = FaceFullSet(TRAIN_DIR, VALID_DIR, VALID_CSV, tfm=aug_s2)
full_dl    = DataLoader(full_ds, batch_size=BATCH_SIZE, shuffle=True,
                        drop_last=True, num_workers=NUM_WORKERS)
val_ds     = ValidDataset(VALID_DIR, VALID_CSV, transform=val_tfm)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

head_p     = [p for n,p in net.named_parameters() if 'classifier' in n]
backbone_p = [p for n,p in net.named_parameters() if 'classifier' not in n]
opt2 = optim.Adam([
    {'params': backbone_p, 'lr': 1e-4},
    {'params': head_p,     'lr': 1e-3},
], weight_decay=5e-4)

loss_fn  = nn.CrossEntropyLoss(label_smoothing=0.1)
ema      = EMA(net, decay=EMA_DECAY)   # NEW: EMA
best_ema = 0.0
best_ep  = 0

print(f'Stage 2: {NUM_EP_S2} epochs | EMA decay={EMA_DECAY}')
print(f"{'Epoch':>6}  {'Loss':>8}  {'TrainAcc':>9}  {'ValAcc':>8}  {'EMA_Val':>8}")
print('-' * 52)

for ep in range(NUM_EP_S2):
    net.train()
    opt2.param_groups[0]['lr'] = lr_schedule(ep, 3, 1e-4, NUM_EP_S2)
    opt2.param_groups[1]['lr'] = lr_schedule(ep, 3, 1e-3, NUM_EP_S2)
    ep_loss, correct, total = 0.0, 0, 0

    for imgs, lbls, _ in full_dl:
        imgs = imgs.to(DEVICE)
        lbls = lbls.clone().detach().long().to(DEVICE)
        opt2.zero_grad()
        logits = net(imgs)
        loss   = loss_fn(logits, lbls)
        loss.backward()
        nn.utils.clip_grad_norm_(net.parameters(), 1.0)
        opt2.step()
        ema.update(net)   # update EMA after every step
        ep_loss += loss.item() * imgs.size(0)
        correct += (logits.argmax(1) == lbls).sum().item()
        total   += imgs.size(0)

    val_acc = evaluate(net, val_loader)

    # Evaluate EMA model
    ema.store(net); ema.apply_to(net)
    ema_acc = evaluate(net, val_loader)
    ema.restore(net)

    marker = ''
    if ema_acc > best_ema:
        best_ema = ema_acc
        best_ep  = ep + 1
        # save EMA weights as best model
        ema.store(net); ema.apply_to(net)
        torch.save(net, f'{ROLL_NO}_best.pth')
        ema.restore(net)
        marker = '  *'

    print(f'{ep+1:6d}  {ep_loss/total:8.4f}  '
          f'{correct/total*100:9.2f}%  {val_acc:8.2f}%  {ema_acc:8.2f}%{marker}')

print(f'\nBest EMA Val Acc: {best_ema:.2f}% at epoch {best_ep}')


Dataset: 18466 images
Stage 2: 80 epochs | EMA decay=0.9997
 Epoch      Loss   TrainAcc    ValAcc   EMA_Val
----------------------------------------------------
     1    0.2607      97.06%     77.61%     47.76%  *
     2    0.2408      98.06%     75.37%     47.76%
     3    0.2435      97.88%     79.85%     47.76%
     4    0.2368      98.12%     73.88%     47.76%
     5    0.2360      98.25%     83.58%     47.76%
     6    0.2354      98.20%     79.10%     49.25%  *
     7    0.2347      98.29%     80.60%     67.16%  *
     8    0.2327      98.31%     85.82%     73.13%  *
     9    0.2286      98.65%     83.58%     73.88%  *
    10    0.2287      98.63%     79.85%     75.37%  *
    11    0.2268      98.68%     85.07%     77.61%  *
    12    0.2279      98.65%     91.79%     76.12%
    13    0.2271      98.63%     87.31%     77.61%
    14    0.2270      98.71%     85.07%     79.85%  *
    15    0.2252      98.80%     85.82%     80.60%  *
    16    0.2251      98.86%     84.33%     82.

In [31]:
# TTA evaluation on best EMA model
best_net = torch.load(f'{ROLL_NO}_best.pth', map_location=DEVICE, weights_only=False)
tta_acc  = evaluate_tta(best_net, VALID_DIR, VALID_CSV)
print(f'Best EMA Val Acc : {best_ema:.2f}%')
print(f'TTA Val Acc      : {tta_acc:.2f}%')

# Save as final submission
torch.save(best_net, f'{ROLL_NO}.pth')
print(f'\nSaved: {ROLL_NO}.pth ({os.path.getsize(ROLL_NO+".pth")/1e6:.1f} MB)')


Best EMA Val Acc : 99.25%
TTA Val Acc      : 99.25%

Saved: b23cm1033.pth (45.4 MB)


In [32]:
# Verify submission
loaded = torch.load(f'{ROLL_NO}.pth', map_location=DEVICE, weights_only=False)
loaded.eval()
with torch.no_grad():
    out = loaded(torch.randn(4,3,224,224).to(DEVICE))
assert out.shape == (4,2)
print(f'Model verified. Shape: {out.shape}')
print(f'Val Acc: {evaluate(loaded, val_loader):.2f}%')
print()
print('Run to verify:')
print(f'  python evaluate_submission_student.py --model_path {ROLL_NO}.pth --model_file {ROLL_NO}.py --data_dir dataset')


Model verified. Shape: torch.Size([4, 2])
Val Acc: 99.25%

Run to verify:
  python evaluate_submission_student.py --model_path b23cm1033.pth --model_file b23cm1033.py --data_dir dataset


In [34]:
import sys, importlib.util

# Load _Net from b23cm1033.py
spec = importlib.util.spec_from_file_location('b23cm1033', 'b23cm1033.py')
mod  = importlib.util.module_from_spec(spec)
sys.modules['b23cm1033'] = mod
spec.loader.exec_module(mod)

# Load old model weights
old_model = torch.load('b23cm1033.pth', map_location=DEVICE, weights_only=False)

# Create new model from b23cm1033.py and copy weights
new_model = mod.build_model().to(DEVICE)
new_model.load_state_dict(old_model.state_dict())

# Save new model — now pickle finds _Net in b23cm1033 module
torch.save(new_model, 'b23cm1033.pth')
print('Resaved successfully!')

# Verify
loaded = torch.load('b23cm1033.pth', map_location=DEVICE, weights_only=False)
loaded.eval()
with torch.no_grad():
    out = loaded(torch.randn(4,3,224,224).to(DEVICE))
print(f'Verified. Shape: {out.shape}')

Resaved successfully!
Verified. Shape: torch.Size([4, 2])


In [35]:
!python evaluate_submission_student.py --model_path b23cm1033.pth --model_file b23cm1033.py --data_dir dataset

Device: cuda
Importing model definitions from: b23cm1033.py
Loading model from: b23cm1033.pth
Running inference on valid images...
  134 images processed.

Accuracy: 99.25%  (133/134)
